# Tokyo PoC
New streamlined version

In [ ]:
import time
import json
import math
import random
import numpy as np
import csv
import os

import poc_startup as startup
from filters import RSSIKalmanFilter, AdaptiveBiasFilter
from rt import manage_location_message, manage_online_reconfiguration, get_path_loss, get_los

sionna_structure = startup.startup()
udp_socket = sionna_structure["udp_socket"]
verbose = sionna_structure["verbose"]
my_cam = sionna_structure["my_cam"]
tx_power = sionna_structure["tx_power"]
correction = sionna_structure["correction"]

if sionna_structure["use_kalman_filter"]:
    sionna_structure["filters"] = {
        ("1","2"): RSSIKalmanFilter(process_var=sionna_structure["kalman_process_var"], meas_var=sionna_structure["kalman_meas_var"], rt_var=sionna_structure["kalman_rt_var"]),
        ("1","3"): RSSIKalmanFilter(process_var=sionna_structure["kalman_process_var"], meas_var=sionna_structure["kalman_meas_var"], rt_var=sionna_structure["kalman_rt_var"]),
        ("2","3"): RSSIKalmanFilter(process_var=sionna_structure["kalman_process_var"], meas_var=sionna_structure["kalman_meas_var"], rt_var=sionna_structure["kalman_rt_var"]),
    }

if sionna_structure["use_adaptive_bias_filter"]:
    sionna_structure["filters"] = {
        ("1","2"): AdaptiveBiasFilter(alpha_signal=sionna_structure["adaptive_bias_alpha_signal"], alpha_bias=sionna_structure["adaptive_bias_alpha_bias"]),
        ("1","3"): AdaptiveBiasFilter(alpha_signal=sionna_structure["adaptive_bias_alpha_signal"], alpha_bias=sionna_structure["adaptive_bias_alpha_bias"]),
        ("2","3"): AdaptiveBiasFilter(alpha_signal=sionna_structure["adaptive_bias_alpha_signal"], alpha_bias=sionna_structure["adaptive_bias_alpha_bias"])
    }


def main():
    while True:
        # Receive data from the socket
        payload, address = udp_socket.recvfrom(1024*10)
        sionna_structure["latest_msg_address"] = address
        message = payload.decode()
        
        t_tot = time.time()
        # Parse the JSON message
        data = json.loads(message)

        if isinstance(data, dict) and "data" in data and isinstance(data["data"], list):
            msg_entries = data["data"]
        elif isinstance(data, list):
            msg_entries = data
        else:
            msg_entries = [data]

        type_val = None
        if len(msg_entries) > 0 and isinstance(msg_entries[0], dict):
            type_val = msg_entries[0].get("type")

        if isinstance(type_val, str) and "RT_CONFIGURATION" in type_val.upper():
            manage_online_reconfiguration(msg_entries, sionna_structure, is_manual_override=False)
            continue

        if isinstance(data, dict) and "data" in data and isinstance(data["data"], list):
            data = data["data"]
        
        if verbose:
            print(" ")
            print("  - - - - - - - - - - - - - - - - -  NEW PREDICTION REQUEST  - - - - - - - - - - - - - - - - -  ")
            print(" ")

        # Extract current measured RSSI to calibrate the Digital Twin
        current_1vs2 = data[0]["data"]["RSSI_Vehicle1_2"] if data[0]["data"]["RSSI_Vehicle1_2"] != 0 else None
        current_1vs3 = data[0]["data"]["RSSI_Vehicle1_3"] if data[0]["data"]["RSSI_Vehicle1_3"] != 0 else None
        current_2vs3 = data[0]["data"]["RSSI_Vehicle2_3"] if data[0]["data"]["RSSI_Vehicle2_3"] != 0 else None
        
        if sionna_structure["use_kalman_filter"]:
            for key, filt in sionna_structure["filters"].items():
                filt.predict()
        

        # Update the locations in the scenario
        future_timestamp = data[1]["clock"]
        predicted_p1 = data[1]["data"]["x1"]
        predicted_p2 = data[1]["data"]["x2"]
        predicted_p3 = data[1]["data"]["x3"]
        t = time.time()

        manage_location_message(f"LOC_UPDATE:veh2,{predicted_p2['x']},{predicted_p2['z']},{0.0},{predicted_p2['yaw']},0,0,0", sionna_structure)
        # Hardcoded for static vehicles
        #manage_location_message(f"LOC_UPDATE:veh3,{predicted_p1['x']},{predicted_p1['z']},{0.0},{predicted_p1['yaw']},0,0,0", sionna_structure) # I know they are inverted!
        #manage_location_message(f"LOC_UPDATE:veh1,{predicted_p3['x']},{predicted_p3['z']},{0.0},{predicted_p3['yaw']},0,0,0", sionna_structure) # I know they are inverted!
        manage_location_message(f"LOC_UPDATE:veh1,{-13467.166},{-43713.134},{0.0},{99.0512},0,0,0", sionna_structure) # fix them
        manage_location_message(f"LOC_UPDATE:veh3,{-13491.5322},{-43709.6992},{0.0},{99.0512},0,0,0", sionna_structure) # fix them
    
        location_update_time = (time.time() - t) * 1000
        if sionna_structure["time_checker"]:
            print(f"        Location update took: {location_update_time:.2f} ms")

        # Compute filtered RSSI predictions
        t_c = time.time()
        raw_rssi_1vs2 = {}
        raw_rssi_1vs3 = {}
        raw_rssi_2vs3 = {}
        jit = sionna_structure["montecarlo_max_position_jitter"]
        
        is_monte_carlo_enabled = sionna_structure["montecarlo_realizations"] > 1 and sionna_structure["montecarlo_max_position_jitter"] > 0.0
        real = sionna_structure["montecarlo_realizations"] if is_monte_carlo_enabled else 1

        for i in range(real):
            if is_monte_carlo_enabled:
                sionna_structure["seed"] = random.randint(0, int(1e6))
            print(f"        RSSI Prediction iteration {i+1}/{real} with seed {sionna_structure['seed']}")
            manage_location_message(f"LOC_UPDATE:veh2,{predicted_p2['x'] + random.uniform(-jit, jit)},{predicted_p2['z'] + random.uniform(-jit, jit)},{0.0},{predicted_p2['yaw']},0,0,0", sionna_structure)
            raw_rssi_1vs2[i] = tx_power - get_path_loss("car_1", "car_2", sionna_structure) + correction - 2.5
            los_1_2 = get_los("car_1", "car_2", sionna_structure)
            raw_rssi_1vs3[i] = tx_power - get_path_loss("car_1", "car_3", sionna_structure) + correction + 7 + 1.80 - 0.25
            los_1_3 = get_los("car_1", "car_3", sionna_structure)
            raw_rssi_2vs3[i] = tx_power - get_path_loss("car_2", "car_3", sionna_structure) + correction
            los_2_3 = get_los("car_2", "car_3", sionna_structure)
            sionna_structure["rays_cache"] = {}  # Clear rays cache to force re-computation
            i += 1

        raw_rssi_1vs2 = sum(raw_rssi_1vs2.values()) / real
        raw_rssi_1vs3 = sum(raw_rssi_1vs3.values()) / real
        raw_rssi_2vs3 = sum(raw_rssi_2vs3.values()) / real

        if sionna_structure["use_kalman_filter"]:
            # Update Kalman filters
            rssi_1vs2 = sionna_structure["filters"][("1","2")].update(z_meas=current_1vs2, z_rt=raw_rssi_1vs2)
            rssi_1vs3 = sionna_structure["filters"][("1","3")].update(z_meas=current_1vs3, z_rt=raw_rssi_1vs3)
            rssi_2vs3 = sionna_structure["filters"][("2","3")].update(z_meas=current_2vs3, z_rt=raw_rssi_2vs3)
        
        if sionna_structure["use_adaptive_bias_filter"]:
            rssi_1vs2 = sionna_structure["filters"][("1","2")].step(predicted_rt=raw_rssi_1vs2, current_meas=current_1vs2)
            rssi_1vs3 = sionna_structure["filters"][("1","3")].step(predicted_rt=raw_rssi_1vs3, current_meas=current_1vs3)
            rssi_2vs3 = sionna_structure["filters"][("2","3")].step(predicted_rt=raw_rssi_2vs3, current_meas=current_2vs3)

        rssi_prediction_time = (time.time() - t_c) * 1000
        if sionna_structure["time_checker"]:
            print(f"        RT and RSSI extraction took: {rssi_prediction_time:.2f} ms")

        if sionna_structure["verbose"]:
            print(f"        Raw predicted RSSI:   1vs2={raw_rssi_1vs2:.2f} dBm, 1vs3={raw_rssi_1vs3:.2f} dBm, 2vs3={raw_rssi_2vs3:.2f} dBm")
            print(f"        Filtered RSSI:        1vs2={rssi_1vs2:.2f} dBm, 1vs3={rssi_1vs3:.2f} dBm, 2vs3={rssi_2vs3:.2f} dBm")

        # Prepare the response
        response = [{}]
        response[0]["clock"] = future_timestamp
        response[0]["data"] = {}
        response[0]["data"]["x1"] = predicted_p1
        response[0]["data"]["x2"] = predicted_p2
        response[0]["data"]["x3"] = predicted_p3
        response[0]["data"]["RSSI_Vehicle1_2"] = rssi_1vs2
        response[0]["data"]["RSSI_Vehicle1_3"] = rssi_1vs3
        response[0]["data"]["RSSI_Vehicle2_3"] = rssi_2vs3
        response[0]["data"]["raw_RSSI_Vehicle1_2"] = raw_rssi_1vs2
        response[0]["data"]["raw_RSSI_Vehicle1_3"] = raw_rssi_1vs3
        response[0]["data"]["raw_RSSI_Vehicle2_3"] = raw_rssi_2vs3

        response = json.dumps(response, default=lambda o: float(o) if isinstance(o, np.float32) else o)

        # Send back the response
        #udp_socket.sendto(response.encode(), ("20.0.0.10", 35944))
        udp_socket.sendto(response.encode(), address) # Use this for local testing script
        total_elapsed_time = (time.time() - t_tot) * 1000

        if verbose:
            print(" ")
            print(f"  - - - - - - - - - - - - - - - - -   Handled in {total_elapsed_time:.2f} ms   - - - - - - - - - - - - - - - - -  ")
            print(" ")
            print(" ")
        
        # Logging
        local_timestamp = time.time()
        with open(sionna_structure["log_file"], mode="a", newline="") as file:
            writer = csv.writer(file)
            writer.writerow([local_timestamp, data[0]["clock"], future_timestamp,
                             message,
                             sionna_structure["sionna_location_db"][1]['x'], sionna_structure["sionna_location_db"][1]['y'], sionna_structure["sionna_location_db"][1]['angle'],
                             sionna_structure["sionna_location_db"][2]['x'], sionna_structure["sionna_location_db"][2]['y'], sionna_structure["sionna_location_db"][2]['angle'],
                             sionna_structure["sionna_location_db"][3]['x'], sionna_structure["sionna_location_db"][3]['y'], sionna_structure["sionna_location_db"][3]['angle'],
                             raw_rssi_1vs2, raw_rssi_1vs3, raw_rssi_2vs3, 
                             rssi_1vs2, rssi_1vs3, rssi_2vs3,
                             location_update_time, rssi_prediction_time, total_elapsed_time, los_1_2, los_1_3, los_2_3])
        
        '''
        from sionna.rt import render_to_file
        # Frame-by-frame exporter
        global frame_num
        print("Exporting frame ", frame_num)
        
        sionna_structure["scene"].render_to_file(camera=my_cam, 
                                         filename=f"/home/rpegurri/Tokyo NDT Integration/PoC Tokyo Science/export/frame_{frame_num}.png", 
                                         show_devices=False)
        frame_num += 1
        '''
        

        if message.startswith("SHUTDOWN_SIONNA"):
            print("Got SHUTDOWN_SIONNA message. Bye!")
            udp_socket.close()
            break

# Entry point
if __name__ == "__main__":
    main()